In [1]:
print("hello")

hello


# Segment level phonological probing



In [ ]:
import os, sys

# Run as if from the project root, regardless of where this notebook lives
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pickle 
import numpy as np
import pandas as pd
from datasets import load_dataset

from src.phonology import (
    phonological_features, text_to_ipa, keep_for, FEATURES, SEGMENT_FILTERS,
)

pd.set_option("display.max_rows", 60)

/home/hpc/vlbi/vlbi108v/miniconda3/envs/feature/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
LANGUAGES = ["en_us", "de_de", "es_419"]
SPLIT = "train"
N_SAMPLES = 100          # per language; keep in sync with what you extracted embeddings for
OUTPUT = "artifacts/phon_features.pkl"
print(FEATURES, "| filters:", SEGMENT_FILTERS)

['voicing', 'nasal', 'manner', 'place'] | filters: {'voicing': 'obstruent', 'nasal': 'all', 'manner': 'consonant', 'place': 'consonant'}


In [ ]:
## 1a. Sanity check


examples = {
    "en_us": "the quick brown fox jumps",
    "de_de": "der schnelle braune Fuchs",
    "es_419": "el rapido zorro marron",
}
for lang in LANGUAGES:
    text = examples[lang]
    print(f"\n=== {lang}: {text!r}")
    print("IPA:", text_to_ipa(text, lang))
    display(pd.DataFrame(phonological_features(text, lang)))


=== en_us: 'the quick brown fox jumps'
IPA: ðəkwˈɪkbɹˈaʊnfˈɑksd͡ʒˈʌmps


,phoneme,voicing,nasal,manner,place,is_vowel,is_obstruent
0,ð,voiced,oral,fricative,coronal,False,True
1,ə,voiced,oral,None,None,True,False
2,k,voiceless,oral,plosive,dorsal,False,True
3,w,voiced,oral,approximant,labial,False,False
4,ɪ,voiced,oral,None,None,True,False
5,k,voiceless,oral,plosive,dorsal,False,True
6,b,voiced,oral,plosive,labial,False,True
7,ɹ,voiced,oral,approximant,coronal,False,False
8,a,voiced,oral,None,None,True,False
9,ʊ,voiced,oral,None,None,True,False



=== de_de: 'der schnelle braune Fuchs'
IPA: deːɐʃnɛləbʁaoːnəfʊks


,phoneme,voicing,nasal,manner,place,is_vowel,is_obstruent
0,d,voiced,oral,plosive,coronal,False,True
1,eː,voiced,oral,None,None,True,False
2,ɐ,voiced,oral,None,None,True,False
3,ʃ,voiceless,oral,fricative,coronal,False,True
4,n,voiced,nasal,nasal,coronal,False,False
5,ɛ,voiced,oral,None,None,True,False
6,l,voiced,oral,approximant,coronal,False,False
7,ə,voiced,oral,None,None,True,False
8,b,voiced,oral,plosive,labial,False,True
9,ʁ,voiced,oral,fricative,dorsal,False,True



=== es_419: 'el rapido zorro marron'
IPA: elrapidoθoromaron


,phoneme,voicing,nasal,manner,place,is_vowel,is_obstruent
0,e,voiced,oral,None,None,True,False
1,l,voiced,oral,approximant,coronal,False,False
2,r,voiced,oral,approximant,coronal,False,False
3,a,voiced,oral,None,None,True,False
4,p,voiceless,oral,plosive,labial,False,True
5,i,voiced,oral,None,None,True,False
6,d,voiced,oral,plosive,coronal,False,True
7,o,voiced,oral,None,None,True,False
8,θ,voiceless,oral,fricative,coronal,False,True
9,o,voiced,oral,None,None,True,False


## Extract over the dataset


In [2]:
PHONEME_CACHE_PATH = "artifacts/phoneme_cache.pkl"
_step1_cache = {}
if os.path.exists(PHONEME_CACHE_PATH):
    with open(PHONEME_CACHE_PATH, "rb") as _f:
        _step1_cache = pickle.load(_f)
    print(f"Using phoneme cache ({len(_step1_cache)} entries) — skipping live gruut calls.")

rows = []
for lang in LANGUAGES:
    ds = load_dataset("google/fleurs", lang, split=SPLIT, trust_remote_code=True)
    ds = ds.select(range(min(N_SAMPLES, len(ds))))
    for utt_id, text in zip(ds["id"], ds["transcription"]):
        key = (lang, utt_id)
        segs = _step1_cache.get(key) or phonological_features(text, lang)
        for idx, seg in enumerate(segs):
            rows.append({"language": lang, "utt_id": utt_id, "idx": idx, **seg})
    print(f"{lang}: done")

df = pd.DataFrame(rows)
print(f"\nTotal phonemes: {len(df)}")
df.head(15)

NameError: name 'os' is not defined

## Label distributions &amp; the segment-filter effect



In [ ]:
print("voicing on ALL segments:")
print(df["voicing"].value_counts(normalize=True).round(3).to_string())
print("\nvoicing on OBSTRUENTS only (what we actually probe):")
print(df[df.is_obstruent]["voicing"].value_counts(normalize=True).
round(3).to_string())

voicing on ALL segments:
voicing
voiced       0.759
voiceless    0.241

voicing on OBSTRUENTS only (what we actually probe):
voicing
voiceless    0.634
voiced       0.366


In [ ]:
for feat in FEATURES:
    mask = df.apply(lambda r: keep_for(feat, r), axis=1)
    sub = df[mask]
    print(f"\n=== {feat}  (filter: {SEGMENT_FILTERS[feat]}, n={len(sub)}) ===")
    print(sub.groupby("language")[feat].value_counts().to_string())


=== voicing  (filter: obstruent, n=582) ===
language  voicing  
de_de     voiceless    144
          voiced        92
en_us     voiceless     90
          voiced        60
es_419    voiceless    135
          voiced        61

=== nasal  (filter: all, n=1642) ===
language  nasal
de_de     oral     535
          nasal     87
en_us     oral     358
          nasal     48
es_419    oral     553
          nasal     61

=== manner  (filter: consonant, n=962) ===
language  manner     
de_de     fricative      126
          plosive        106
          nasal           87
          approximant     55
          affricate        4
en_us     plosive         81
          fricative       65
          nasal           48
          approximant     46
          affricate        4
es_419    plosive        117
          approximant     83
          fricative       79
          nasal           61

=== place  (filter: consonant, n=962) ===
language  place  
de_de     coronal    221
          dorsal      8

In [ ]:

## Save

os.makedirs(os.path.dirname(OUTPUT) or ".", exist_ok=True)
df.to_pickle(OUTPUT)
print(f"Saved {len(df)} phoneme rows to {OUTPUT}")

Saved 1642 phoneme rows to artifacts/phon_features.pkl


## Step 2 — Build segment-level (X, y) from the embeddings

We now attach each phoneme label to the wav2vec2 frames it occupies. The accurate way is forced alignment (`src/align.py`, MMS) — but that needs the raw audio. To stay self-contained (only the saved `.pkl`s, no audio/download), we use **uniform alignment**: split each utterance's `T` frames evenly across its phonemes and mean-pool each phoneme's slice.

It's approximate (real phonemes vary in duration), so the numbers are a bit noisy/pessimistic — but it lets us run the entire pipeline now. Swapping in `align.py` later only sharpens the frame boundaries; nothing else changes.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
from src.probing import evaluate_probe, cross_lingual_probe

# Embeddings from the extraction jobs: embeddings/<tag>/<lang>_features.pkl
MODELS = {
    "base":     "embeddings/base",
    "xlsr53":   "embeddings/xlsr53",
    "xlsr300m": "embeddings/xlsr300m",
}
MODELS = {k: v for k, v in MODELS.items() if os.path.isdir(v)}
assert MODELS, "No embeddings/<tag> folders found - point MODELS at your extracted embeddings."
print("Models found:", list(MODELS))

N_PROBE = 30      # utterances per language to probe (raise for less noisy numbers)

# ---------------------------------------------------------------------------
# Phoneme cache — gruut takes ~2-3 s/utterance; we persist results to disk.
# Run `python src/precompute_phonemes.py` once to build artifacts/phoneme_cache.pkl
# (~10-15 min, 100 utts × 3 langs). Falls back to live gruut if a key is missing.
# ---------------------------------------------------------------------------
PHONEME_CACHE_PATH = "artifacts/phoneme_cache.pkl"
_phon_cache = {}
if os.path.exists(PHONEME_CACHE_PATH):
    with open(PHONEME_CACHE_PATH, "rb") as _f:
        _phon_cache = pickle.load(_f)
    print(f"Loaded phoneme cache: {len(_phon_cache)} entries from {PHONEME_CACHE_PATH}")
else:
    print(f"WARNING: {PHONEME_CACHE_PATH} not found — utt_phonemes will call gruut live (~3s/utt).")
    print("Run `python src/precompute_phonemes.py` to build the cache first.")


def load_utts(model_dir, lang, n):
    with open(f"{model_dir}/{lang}_features.pkl", "rb") as f:
        return pickle.load(f)[:n]


def utt_phonemes(sample, lang):
    key = (lang, sample.get("id"))
    if key not in _phon_cache:
        _phon_cache[key] = phonological_features(sample.get("transcription", ""), lang)
    return _phon_cache[key]


def segment_xy(utts_by_lang, layer, feature):
    """Uniform alignment: split each utterance's frames evenly across its phonemes,
    mean-pool each phoneme's slice at `layer`, keep only segments relevant to
    `feature`. Returns (X, y) pooled over the given languages."""
    X, y = [], []
    for lang, utts in utts_by_lang.items():
        for s in utts:
            segs = utt_phonemes(s, lang)
            if not segs:
                continue
            h = s["hidden_states"][layer][0]          # (T, D)
            T, n = len(h), len(segs)
            for i, seg in enumerate(segs):
                if not keep_for(feature, seg):
                    continue
                f0 = int(i / n * T)
                f1 = max(f0 + 1, int((i + 1) / n * T))
                X.append(h[f0:f1].mean(axis=0))
                y.append(seg[feature])
    return np.array(X), np.array(y)


def safe_evaluate(X, y):
    if len(y) < 12 or len(set(y)) < 2 or min(Counter(y).values()) < 2:
        return None
    try:
        return evaluate_probe(X, y)
    except Exception:
        return None

ModuleNotFoundError: No module named 'src'

## Step 3 — Run the probes

The compute cell: for every model × feature it builds `(X, y)` by uniform alignment and runs the linear probe. It collects three tables — `layer_df` (per-layer, within-language), `within_df` (reference at the middle layer), and `xling_df` (cross-lingual transfer). With few samples, cells with too few examples per class are skipped automatically.

In [ ]:
layer_rows, within_rows, xling_rows = [], [], []

for tag, mdir in MODELS.items():
    print(f"== {tag} ==")
    utts = {lang: load_utts(mdir, lang, N_PROBE) for lang in LANGUAGES}
    n_layers = len(utts["en_us"][0]["hidden_states"])
    layers = sorted({int(r * (n_layers - 1)) for r in (0, 0.25, 0.5, 0.75, 1.0)})
    probe_layer = n_layers // 2

    for feat in FEATURES:
        # H2: layer analysis, within-language (pooled across languages)
        for L in layers:
            res = safe_evaluate(*segment_xy(utts, L, feat))
            if res:
                layer_rows.append({"model": tag, "feature": feat, "layer": L, **res})

        # within-language reference at the probe layer (for H3)
        rw = safe_evaluate(*segment_xy(utts, probe_layer, feat))
        if rw:
            within_rows.append({"model": tag, "feature": feat, "layer": probe_layer, **rw})

        # H1/H3: cross-lingual transfer (train EN -> test DE/ES) at the probe layer
        Xen, yen = segment_xy({"en_us": utts["en_us"]}, probe_layer, feat)
        for test in ["de_de", "es_419"]:
            Xt, yt = segment_xy({test: utts[test]}, probe_layer, feat)
            if len(yen) >= 12 and len(yt) >= 6 and len(set(yen)) >= 2:
                r = cross_lingual_probe(Xen, yen, Xt, yt)
                xling_rows.append({"model": tag, "feature": feat, "test": test,
                                   "layer": probe_layer, **r})
    del utts

layer_df = pd.DataFrame(layer_rows)
within_df = pd.DataFrame(within_rows)
xling_df = pd.DataFrame(xling_rows)
print("\nlayer:", layer_df.shape, "| within:", within_df.shape, "| xling:", xling_df.shape)

== base ==


: 

## Step 4a — H2: which layers encode phonology best

Within-language decodability (macro-F1) by layer, one line per feature. H2 predicts a **non-flat curve peaking in the lower-middle layers** (where phonetic detail lives) rather than flat or top-heavy.

In [ ]:
mods = list(MODELS)
fig, axes = plt.subplots(1, len(mods), figsize=(5 * len(mods), 4), squeeze=False, sharey=True)
for ax, tag in zip(axes[0], mods):
    sub = layer_df[layer_df.model == tag]
    for feat in FEATURES:
        s = sub[sub.feature == feat].sort_values("layer")
        if len(s):
            ax.plot(s.layer, s.macro_f1, marker="o", label=feat)
    ax.set_title(tag); ax.set_xlabel("layer"); ax.set_ylabel("macro-F1")
axes[0][0].legend(fontsize=8)
fig.suptitle("H2: phonological feature decodability by layer", y=1.03)
fig.tight_layout(); plt.show()

layer_df.pivot_table(index=["model", "layer"], columns="feature", values="macro_f1").round(2)

## Step 4b — H1: multilingual vs monolingual

Zero-shot cross-lingual transfer — train the probe on **English**, test on **German & Spanish** — at the middle layer, macro-F1, by model. H1 predicts the multilingual models (`xlsr53`, `xlsr300m`) transfer better than monolingual `base`. Compare each value to its majority baseline below.

In [ ]:
h1 = xling_df.groupby(["feature", "model"]).agg(
    macro_f1=("macro_f1", "mean"), majority=("majority", "mean")).reset_index()

print("Cross-lingual transfer macro-F1 (train EN -> test DE & ES), by model:")
display(h1.pivot(index="feature", columns="model", values="macro_f1").round(3))
print("\nMajority baseline to beat:")
display(h1.pivot(index="feature", columns="model", values="majority").round(3))

## Step 4c — H3: which features/languages are universal

`transfer_gap = within_lang − cross_lang`: a large gap means the feature is **language-specific** (transfers poorly across languages). H3 predicts **place** has the largest gap, and **voicing/nasal** the smallest.

In [ ]:
w = within_df.groupby("feature")["macro_f1"].mean()
c = xling_df.groupby("feature")["macro_f1"].mean()
h3 = pd.DataFrame({"within_lang": w, "cross_lang": c})
h3["transfer_gap"] = h3.within_lang - h3.cross_lang
h3.sort_values("transfer_gap", ascending=False).round(3)

## Reading the results (and caveats)

- **H1 (models):** higher cross-lingual macro-F1 for `xlsr53`/`xlsr300m` than `base` supports it.
- **H2 (layers):** a non-flat curve peaking in the early–middle layers supports it.
- **H3 (features):** voicing/nasal transfer well (small `transfer_gap`); place poorly (large gap).

**This is a small, fast run — read it as a working pipeline, not final numbers:**
- Only `N_PROBE` utterances/language and **uniform alignment** (not forced) → noisy, slightly pessimistic. Raise `N_PROBE` and/or swap in `src/align.py` (MMS, needs audio) for real results.
- Always compare a cross-lingual score to its **majority baseline** — a score near the baseline means *no* real transfer.
- If the large models run out of memory while loading, lower `N_PROBE` or run `base` alone first.